# Proyecto FIFA World Cup 2022  
## Formativa 3 — Fase 3: Notebook con scripts y mediciones de complejidad

**Asignatura:** MCDI500 — Herramientas de Software Científico  
**Evaluación:** Formativa 3 — Avance Fase 3, Semana 2  
**Integrantes:** Enzo Pinilla, Claudio Alarcón y Luis Espinoza  
**Docente:** Dr. Omar Salinas Silva  
**Fecha:** 15/06/2026

---

Este notebook construye el avance técnico de Fase 3 a partir del dataset trabajado en Fase 2. El foco no es repetir el pipeline completo de limpieza, sino reutilizar la base procesada y diseñar algoritmos estructurados y recursivos en Python, incorporando mediciones básicas de complejidad temporal y una interpretación técnica de los resultados.

## Índice

1. Contexto y objetivo del avance  
2. Relación con Fase 2 y foro técnico de Semana 1  
3. Configuración del entorno e importación de módulos  
4. Carga del dataset procesado y preprocesamiento mínimo  
5. Diseño algorítmico: entrada, proceso y salida  
6. Algoritmo estructurado iterativo  
7. Algoritmo recursivo  
8. Algoritmo vectorizado con pandas  
9. Validación técnica y verificación de resultados  
10. Medición de complejidad temporal y espacial  
11. Comparación e interpretación  
12. Arquitectura del código y proyección  
13. Bibliografía

## 1. Contexto y objetivo del avance

En la Fase 2 se construyó un pipeline de obtención, limpieza y transformación de datos para el dataset **FIFA World Cup 2022**. En esta Fase 3 se utiliza esa base para analizar decisiones de diseño algorítmico.

**Objetivo del notebook:** implementar y comparar prototipos algorítmicos estructurados, recursivos y vectorizados para calcular estadísticas simples del torneo, verificando la coherencia entre datos de entrada, procesamiento y resultados esperados.

El problema elegido corresponde al cálculo del **total de goles registrados en los partidos** y al ordenamiento de selecciones según goles acumulados. Este problema permite comparar recorridos iterativos, recursividad y funciones optimizadas de Python/pandas.

## 2. Relación con Fase 2 y foro técnico de Semana 1

Este notebook reutiliza el dataset trabajado en Fase 2. Por lo tanto, solo se aplican transformaciones mínimas necesarias para el análisis algorítmico: Estandarización de nombres de columnas, verificación de columnas requeridas y conversión numérica de las columnas de goles.

En coherencia con el foro técnico de la Semana 1, se prioriza la programación estructurada para operaciones tabulares simples, porque ofrece mayor legibilidad y menor uso adicional de memoria. La recursividad se incluye como comparación técnica y como evidencia de diseño algorítmico, pero se analiza críticamente su costo espacial asociado a la pila de llamadas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import timeit

# Permite importar módulos desde src/ si el notebook se ejecuta desde notebooks/ o desde la raíz del repo.
SRC_PATH = Path("../src").resolve()
if not SRC_PATH.exists():
    SRC_PATH = Path("src").resolve()

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from funciones_reutilizables import cargar_datos, normalizar_nombres_columnas
from funciones_f3_algoritmos import (
    preparar_lista_goles,
    suma_iterativa,
    suma_recursiva,
    suma_vectorizada_pandas,
    obtener_goles_por_equipo,
    merge_sort_recursivo,
    ordenar_builtin,
    medir_tiempo,
    estimar_memoria_lista,
)

np.random.seed(42)
print("Entorno configurado correctamente")

Entorno configurado correctamente


## 3. Carga del dataset crudo y preprocesamiento mínimo

Se carga el dataset original desde `data/raw/Fifa_world_cup_matches.csv`, porque esta fase necesita las columnas nominales `team1` y `team2` para agrupar goles por selección. El archivo procesado de Fase 2 ya tiene esas variables codificadas con One-Hot Encoding.

La transformación mínima consiste en normalizar nombres de columnas y asegurar que las columnas de goles estén en formato numérico. No se rehace el pipeline completo de Fase 2 porque esta actividad se centra en algoritmos y complejidad.

In [ ]:
rutas_posibles = [
    Path("../data/raw/Fifa_world_cup_matches.csv"),
    Path("data/raw/Fifa_world_cup_matches.csv"),
]

ultima_excepcion = None
for ruta in rutas_posibles:
    try:
        df = cargar_datos(ruta)
        print(f"Archivo utilizado: {ruta}")
        break
    except FileNotFoundError as e:
        ultima_excepcion = e
else:
    raise ultima_excepcion

# F3 necesita las columnas nominales team1/team2; el dataset procesado ya las trae codificadas.
df = normalizar_nombres_columnas(df)
print("Dimensiones del dataset:", df.shape)
df.head()

Datos cargados: 64 filas y 88 columnas.
Archivo utilizado: ../data/raw/Fifa_world_cup_matches.csv
Dimensiones del dataset: (64, 88)


,team1,team2,possession team1,possession team2,possession in contest,number of goals team1,number of goals team2,date,hour,category,...,penalties scored team1,penalties scored team2,goal preventions team1,goal preventions team2,own goals team1,own goals team2,forced turnovers team1,forced turnovers team2,defensive pressures applied team1,defensive pressures applied team2
0,QATAR,ECUADOR,42%,50%,8%,0,2,20 NOV 2022,17 : 00,Group A,...,0,1,6,5,0,0,52,72,256,279
1,ENGLAND,IRAN,72%,19%,9%,6,2,21 NOV 2022,14 : 00,Group B,...,0,1,8,13,0,0,63,72,139,416
2,SENEGAL,NETHERLANDS,44%,45%,11%,0,2,21 NOV 2022,17 : 00,Group A,...,0,0,9,15,0,0,63,73,263,251
3,UNITED STATES,WALES,51%,39%,10%,1,1,21 NOV 2022,20 : 00,Group B,...,0,1,7,7,0,0,81,72,242,292
4,ARGENTINA,SAUDI ARABIA,64%,24%,12%,1,2,22 NOV 2022,11 : 00,Group C,...,1,0,4,14,0,0,65,80,163,361


In [ ]:
columnas_requeridas = ["team1", "team2", "number of goals team1", "number of goals team2"]
faltantes = [col for col in columnas_requeridas if col not in df.columns]

if faltantes:
    raise KeyError(f"Faltan columnas requeridas para el análisis algorítmico: {faltantes}")

for col in ["number of goals team1", "number of goals team2"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("Columnas requeridas disponibles y convertidas correctamente.")
df[columnas_requeridas].head()

Columnas requeridas disponibles y convertidas correctamente.


,team1,team2,number of goals team1,number of goals team2
0,QATAR,ECUADOR,0,2
1,ENGLAND,IRAN,6,2
2,SENEGAL,NETHERLANDS,0,2
3,UNITED STATES,WALES,1,1
4,ARGENTINA,SAUDI ARABIA,1,2


## 4. Diseño algorítmico: entrada, proceso y salida

El flujo de datos para esta fase se define de la siguiente forma:

```text
Dataset FIFA procesado en F2
        ↓
selección de columnas de goles y equipos
        ↓
preparar_lista_goles(df)
        ↓
suma_iterativa(datos) / suma_recursiva(datos) / suma_vectorizada_pandas(df)
        ↓
validación de resultados equivalentes
        ↓
medición con timeit e interpretación de complejidad
```

La entrada principal es una lista de goles por equipo en cada partido. La salida esperada es el total de goles del torneo, calculado por distintas estrategias para comparar eficiencia y legibilidad.

In [ ]:
goles = preparar_lista_goles(df)

print("Cantidad de registros de goles:", len(goles))
print("Primeros 10 valores:", goles[:10])
print("Memoria aproximada de la lista de goles en bytes:", estimar_memoria_lista(goles))

Cantidad de registros de goles: 128
Primeros 10 valores: [0, 6, 0, 1, 1, 0, 0, 4, 0, 1]
Memoria aproximada de la lista de goles en bytes: 4504


## 5. Algoritmo estructurado iterativo

La versión iterativa recorre la lista de goles mediante un ciclo `for`. Es una solución clara, mantenible y adecuada para colecciones tabulares simples.

- Complejidad temporal esperada: **O(n)**, porque recorre todos los elementos una vez.
- Complejidad espacial adicional esperada: **O(1)**, porque solo mantiene una variable acumuladora.

In [ ]:
resultado_iterativo = suma_iterativa(goles)
print("Total de goles calculado con versión iterativa:", resultado_iterativo)

Total de goles calculado con versión iterativa: 172


## 6. Algoritmo recursivo

La versión recursiva resuelve el mismo problema mediante un caso base y llamadas sucesivas a la misma función. Se implementa con un índice para evitar copias de lista en cada llamada.

- Complejidad temporal esperada: **O(n)**.
- Complejidad espacial adicional esperada: **O(n)**, por la pila de llamadas recursivas.

Aunque cumple con el objetivo académico de aplicar recursividad, para este caso tabular la versión iterativa suele ser más conveniente en memoria y legibilidad.

In [ ]:
resultado_recursivo = suma_recursiva(goles)
print("Total de goles calculado con versión recursiva:", resultado_recursivo)

Total de goles calculado con versión recursiva: 172


## 7. Algoritmo vectorizado con pandas

La versión vectorizada utiliza operaciones internas de pandas. En análisis de datos tabulares, este enfoque suele ser más eficiente y expresivo, porque evita bucles explícitos en Python y aprovecha implementaciones optimizadas.

In [ ]:
resultado_vectorizado = suma_vectorizada_pandas(df)
print("Total de goles calculado con pandas:", resultado_vectorizado)

Total de goles calculado con pandas: 172


## 8. Validación técnica y verificación de resultados

Se verifica que las tres implementaciones produzcan el mismo resultado. Esta validación básica permite comprobar la coherencia entre entrada, procesamiento y salida esperada. **(pandas development team, 2024)** 

In [ ]:
print("Iterativo:", resultado_iterativo)
print("Recursivo:", resultado_recursivo)
print("Vectorizado:", resultado_vectorizado)

assert resultado_iterativo == resultado_recursivo == resultado_vectorizado
assert len(goles) == df.shape[0] * 2
assert resultado_iterativo >= 0

print("[OK] Las implementaciones entregan resultados coherentes.")

Iterativo: 172
Recursivo: 172
Vectorizado: 172
[OK] Las implementaciones entregan resultados coherentes.


## 9. Medición de complejidad temporal

Se utiliza `timeit` para medir de forma reproducible el tiempo de ejecución de cada implementación. Las repeticiones permiten comparar tendencias generales; los valores exactos pueden variar según el equipo y el entorno de ejecución.
 **(Python Software Foundation, 2024).**

In [ ]:
REPETICIONES = 1000

t_iterativo = medir_tiempo(lambda: suma_iterativa(goles), repeticiones=REPETICIONES)
t_recursivo = medir_tiempo(lambda: suma_recursiva(goles), repeticiones=REPETICIONES)
t_vectorizado = medir_tiempo(lambda: suma_vectorizada_pandas(df), repeticiones=REPETICIONES)

resultados_tiempo = pd.DataFrame({
    "implementacion": ["iterativa", "recursiva", "vectorizada_pandas"],
    "tiempo_segundos": [t_iterativo, t_recursivo, t_vectorizado],
    "repeticiones": [REPETICIONES, REPETICIONES, REPETICIONES],
    "complejidad_temporal_esperada": ["O(n)", "O(n)", "O(n) sobre columnas"],
    "complejidad_espacial_adicional": ["O(1)", "O(n)", "O(1) conceptual"]
})

resultados_tiempo

,implementacion,tiempo_segundos,repeticiones,complejidad_temporal_esperada,complejidad_espacial_adicional
0,iterativa,0.002896,1000,O(n),O(1)
1,recursiva,0.016556,1000,O(n),O(n)
2,vectorizada_pandas,0.016629,1000,O(n) sobre columnas,O(1) conceptual


## 10. Algoritmo recursivo adicional: ordenamiento por goles acumulados

Para evidenciar una estrategia tipo **divide and conquer**, se implementa `merge_sort_recursivo` sobre una lista de pares `(equipo, goles)`. Luego se compara con `sorted`, que corresponde a una implementación optimizada del lenguaje Python.
**(McKinney, 2022; Downey, 2015)**

In [ ]:
goles_por_equipo = obtener_goles_por_equipo(df)
print("Cantidad de equipos:", len(goles_por_equipo))
print("Primeros pares equipo-goles:", goles_por_equipo[:5])

orden_merge = merge_sort_recursivo(goles_por_equipo)
orden_builtin = ordenar_builtin(goles_por_equipo)

print("Top 5 con merge sort recursivo:", orden_merge[:5])
print("Top 5 con sorted:", orden_builtin[:5])

assert orden_merge == orden_builtin
print("[OK] El ordenamiento recursivo produce el mismo resultado que sorted.")

Cantidad de equipos: 32
Primeros pares equipo-goles: [('ARGENTINA', 15), ('AUSTRALIA', 4), ('BELGIUM', 1), ('BRAZIL', 8), ('CAMEROON', 4)]
Top 5 con merge sort recursivo: [('FRANCE', 16), ('ARGENTINA', 15), ('ENGLAND', 13), ('PORTUGAL', 12), ('NETHERLANDS', 10)]
Top 5 con sorted: [('FRANCE', 16), ('ARGENTINA', 15), ('ENGLAND', 13), ('PORTUGAL', 12), ('NETHERLANDS', 10)]
[OK] El ordenamiento recursivo produce el mismo resultado que sorted.


In [ ]:
t_merge = medir_tiempo(lambda: merge_sort_recursivo(goles_por_equipo), repeticiones=1000)
t_builtin = medir_tiempo(lambda: ordenar_builtin(goles_por_equipo), repeticiones=1000)

comparacion_ordenamiento = pd.DataFrame({
    "implementacion": ["merge_sort_recursivo", "sorted_python"],
    "tiempo_segundos": [t_merge, t_builtin],
    "repeticiones": [1000, 1000],
    "complejidad_temporal_esperada": ["O(n log n)", "O(n log n)"],
    "observacion": ["Implementación académica recursiva", "Implementación optimizada del lenguaje"]
})

comparacion_ordenamiento

,implementacion,tiempo_segundos,repeticiones,complejidad_temporal_esperada,observacion
0,merge_sort_recursivo,0.035572,1000,O(n log n),Implementación académica recursiva
1,sorted_python,0.002166,1000,O(n log n),Implementación optimizada del lenguaje


## 11. Interpretación de resultados y oportunidades de optimización

La medición permite fundamentar la decisión técnica en evidencia y no solo en supuestos. Para el cálculo del total de goles, la versión iterativa y la recursiva tienen complejidad temporal O(n), pero la recursiva requiere memoria adicional por la pila de llamadas. Por esta razón, en problemas tabulares simples, la recursividad no representa una mejora práctica.

La versión vectorizada con pandas es la alternativa más apropiada para el análisis de datos, ya que expresa directamente la operación sobre columnas y aprovecha optimizaciones internas de la librería. En el caso del ordenamiento, `merge_sort_recursivo` permite evidenciar divide and conquer, pero `sorted` es preferible en un sistema real por estar optimizado y probado.

**Mejora aplicada:** la función recursiva de suma usa un índice en lugar de `datos[1:]`, evitando crear copias de listas en cada llamada. Esta decisión reduce el uso innecesario de memoria respecto de una recursión basada en slicing.

## 12. Documentación de arquitectura del código

| Componente | Responsabilidad | Justificación técnica |
|---|---|---|
| `F3_Definicion.ipynb` | Orquestar la ejecución, mostrar resultados, interpretar complejidad y documentar decisiones. | El notebook facilita reproducibilidad y evidencia narrativa del avance. |
| `src/funciones_reutilizables.py` | Reutilizar funciones de Fase 2, como carga y normalización de datos. | Evita duplicar lógica ya desarrollada en el pipeline anterior. |
| `src/funciones_f3_algoritmos.py` | Concentrar algoritmos estructurados, recursivos, vectorizados y mediciones. | Separa responsabilidades y permite reutilizar/probar funciones fuera del notebook. |
| `data/processed/` | Almacenar el dataset procesado desde Fase 2. | Mantiene continuidad entre F2 y F3 sin repetir todo el preprocesamiento. |

La arquitectura actual es modular inicial. El notebook actúa como documento ejecutable y los scripts contienen la lógica reutilizable. En una fase posterior, esta organización puede evolucionar hacia clases simples o módulos especializados para análisis, medición de rendimiento y visualización.

## 13. Bibliografía

- Downey, A. B. (2015). *Think Python: How to Think Like a Computer Scientist* (2nd ed.). Green Tea Press. https://greenteapress.com/wp/think-python-2e/
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O’Reilly Media.
- pandas development team. (2024). *pandas documentation*. https://pandas.pydata.org/docs/
- Python Software Foundation. (2024). *The Python standard library: timeit*. https://docs.python.org/3/library/timeit.html
- Python Software Foundation. (2024). *Data structures*. https://docs.python.org/3/tutorial/datastructures.html